# RC Model Validation: Python vs. MATLAB

**Objective:** Validate the Python RC model implementation against MATLAB reference.

**Status:** Model corrections applied:
1. ✓ Hour-of-day indexing for schedules: `(hour_counter - 1) % 24`
2. ✓ Config parameters: infiltration/ventilation rates corrected
3. ✓ Equipment schedule: synced with MATLAB

**Expected Results:** Heating ≈ 54 MWh, Cooling ≈ -0.3 MWh


In [1]:
import os, sys
import numpy as np
import pandas as pd
from scipy.io import loadmat
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from core.bootstrap import create_facade

PROJECT_ID = "rc-model-validation"
VARIANT_ID = "Val"
print("✓ Setup complete")

✓ Setup complete


In [2]:
# Run simulation and load all data
facade_Val = create_facade(PROJECT_ID, VARIANT_ID)
facade_Val.run_simulation(PROJECT_ID, VARIANT_ID)
df_res_py = facade_Val._result.load_raw().drop(columns=["temperature_outdoor_air"], axis=1)

# Load MATLAB reference
mat_path = Path(PROJECT_ROOT) / 'projects' / 'rc-model-validation' / 'mat-reference' / 'matlab_ref_results.mat'
mat_data = loadmat(mat_path, squeeze_me=True)
df_res_mat = pd.DataFrame({
    'output_heating_power': mat_data['output_heating_power'],
    'output_cooling_power': mat_data['output_cooling_power'],
    'output_lighting_electricity': mat_data['output_lighting_electricity'],
    'output_equipment_electricity': mat_data['output_equipment_electricity']
})
temp_mat_array = mat_data['output_temperatures']
row_names_all = df_res_py.columns.tolist()
row_names = row_names_all[:-4]
df_temp_outputs = pd.DataFrame(data=temp_mat_array, columns=row_names)
df_res_mat = pd.concat([df_temp_outputs, df_res_mat], axis=1)
df_res_mat.index = df_res_py.index

# Load IDA ICE reference
ida_path = Path(PROJECT_ROOT) / 'projects' / 'rc-model-validation' / 'mat-reference' / 'ida_results_ver2.mat'
ida_data = loadmat(ida_path, squeeze_me=True)
df_res_ida = pd.DataFrame({
    "temperature_air_room": ida_data['ida_air_temp'],
    "output_heating_power": ida_data['ida_heating_power'],
    "output_cooling_power": ida_data['ida_cooling_power'],
})
df_res_ida.index = df_res_py.index
print("✓ Simulation complete. All data loaded.")

Simulation finished.
✓ Simulation complete. All data loaded.


In [3]:
# Validate
df_diff = df_res_py.subtract(df_res_mat, fill_value=0)
df_diff_ida = df_res_py[df_res_ida.columns].subtract(df_res_ida, fill_value=0)

print("\n" + "="*90)
print("VALIDATION: Python vs. MATLAB vs. IDA ICE")
print("="*90)

# Temperature
temp_diff = df_diff["temperature_air_room"]
temp_max = np.max(np.abs(temp_diff))
temp_rmse = np.sqrt(np.mean(temp_diff**2))
print(f"\nTEMPERATURE (°C)")
print(f"  Max Error:  {temp_max:>8.4f}°C  |  RMSE: {temp_rmse:>8.4f}°C  |  Mean: {np.mean(temp_diff):>+8.4f}°C")

# Heating
heat_diff = df_diff["output_heating_power"]
py_heat = df_res_py['output_heating_power'].sum()/1e6
ml_heat = df_res_mat['output_heating_power'].sum()/1e6
heat_pct = ((py_heat/ml_heat-1)*100) if ml_heat != 0 else 0
print(f"\nHEATING (MWh)")
print(f"  Python: {py_heat:>8.2f}  |  MATLAB: {ml_heat:>8.2f}  |  Diff: {py_heat-ml_heat:>+8.2f} ({heat_pct:>+6.1f}%)")

# Cooling
cool_diff = df_diff["output_cooling_power"]
py_cool = df_res_py['output_cooling_power'].sum()/1e6
ml_cool = df_res_mat['output_cooling_power'].sum()/1e6
print(f"\nCOOLING (MWh)")
print(f"  Python: {py_cool:>8.2f}  |  MATLAB: {ml_cool:>8.2f}  |  Diff: {py_cool-ml_cool:>+8.2f}")

# IDA comparison
temp_diff_ida = df_diff_ida["temperature_air_room"]
temp_rmse_ida = np.sqrt(np.mean(temp_diff_ida**2))
print(f"\nPython vs. IDA ICE")
print(f"  Temp RMSE: {temp_rmse_ida:>8.4f}°C")

# Summary
print("\n" + "="*90)
if temp_max < 0.5 and abs(heat_pct) < 1.0:
    print("✓✓✓ VALIDATION SUCCESSFUL")
elif temp_max < 1.0 and abs(heat_pct) < 5.0:
    print("✓ VALIDATION ACCEPTABLE")
else:
    print("⚠ Discrepancies remain - further investigation needed")
print("="*90)


VALIDATION: Python vs. MATLAB vs. IDA ICE

TEMPERATURE (°C)
  Max Error:    1.0432°C  |  RMSE:   0.1856°C  |  Mean:  -0.0020°C

HEATING (MWh)
  Python:    56.61  |  MATLAB:    53.93  |  Diff:    +2.68 (  +5.0%)

COOLING (MWh)
  Python:    -0.55  |  MATLAB:    -0.28  |  Diff:    -0.27

Python vs. IDA ICE
  Temp RMSE:   0.4306°C

⚠ Discrepancies remain - further investigation needed


In [4]:
# Diagnostics: energy totals and first-winter-week diff
# Totals (MWh)
energies = {
    "py_heating_MWh": df_res_py['output_heating_power'].sum()/1e6,
    "py_cooling_MWh": df_res_py['output_cooling_power'].sum()/1e6,
    "py_lighting_MWh": df_res_py['output_lighting_electricity'].sum()/1e6,
    "py_equipment_MWh": df_res_py['output_equipment_electricity'].sum()/1e6,
    "mat_heating_MWh": df_res_mat['output_heating_power'].sum()/1e6,
    "mat_cooling_MWh": df_res_mat['output_cooling_power'].sum()/1e6,
    "mat_lighting_MWh": df_res_mat['output_lighting_electricity'].sum()/1e6,
    "mat_equipment_MWh": df_res_mat['output_equipment_electricity'].sum()/1e6,
}
print("\nENERGY TOTALS (MWh)")
for k,v in energies.items():
    print(f"  {k:20s}: {v:8.3f}")

# First winter week (hours 0-167)
wk = slice(0, 168)
print("\nFIRST WINTER WEEK (hours 0-167):")
print(f"  Heating diff sum (MWh): {(df_diff['output_heating_power'].iloc[wk].sum()/1e6):+.3f}")
print(f"  Cooling diff sum (MWh): {(df_diff['output_cooling_power'].iloc[wk].sum()/1e6):+.3f}")
print(f"  Temp RMSE (°C): {np.sqrt(np.mean(df_diff['temperature_air_room'].iloc[wk]**2)):.4f}")



ENERGY TOTALS (MWh)
  py_heating_MWh      :   56.614
  py_cooling_MWh      :   -0.546
  py_lighting_MWh     :    3.586
  py_equipment_MWh    :   16.768
  mat_heating_MWh     :   53.933
  mat_cooling_MWh     :   -0.277
  mat_lighting_MWh    :    3.402
  mat_equipment_MWh   :   16.768

FIRST WINTER WEEK (hours 0-167):
  Heating diff sum (MWh): +0.089
  Cooling diff sum (MWh): +0.000
  Temp RMSE (°C): 0.0092


In [5]:
# Diagnostics: monthly breakdown and selected weeks
# build a datetime index for grouping
idx = pd.date_range("2020-01-01", periods=len(df_res_py), freq="H")
py = df_res_py.copy(); py.index = idx
mat = df_res_mat.copy(); mat.index = idx

monthly = pd.DataFrame({
    "py_heat_MWh": py['output_heating_power'].resample('M').sum()/1e6,
    "ml_heat_MWh": mat['output_heating_power'].resample('M').sum()/1e6,
    "py_cool_MWh": py['output_cooling_power'].resample('M').sum()/1e6,
    "ml_cool_MWh": mat['output_cooling_power'].resample('M').sum()/1e6,
    "py_temp_mean": py['temperature_air_room'].resample('M').mean(),
    "ml_temp_mean": mat['temperature_air_room'].resample('M').mean(),
})
monthly['heat_diff'] = monthly['py_heat_MWh'] - monthly['ml_heat_MWh']
monthly['cool_diff'] = monthly['py_cool_MWh'] - monthly['ml_cool_MWh']

print("\nMONTHLY HEAT/COOL (MWh)")
print(monthly[['py_heat_MWh','ml_heat_MWh','heat_diff','py_cool_MWh','ml_cool_MWh','cool_diff']].round(2))

# Winter week (Jan 1-7) and summer week (Jul 1-7)
winter = slice("2020-01-01","2020-01-07 23:00")
summer = slice("2020-07-01","2020-07-07 23:00")

for label, sl in [("WINTER_WEEK", winter), ("SUMMER_WEEK", summer)]:
    h_diff = df_diff.loc[:, 'output_heating_power']
    c_diff = df_diff.loc[:, 'output_cooling_power']
    t_diff = df_diff.loc[:, 'temperature_air_room']
    h_sum = h_diff[sl].sum()/1e6
    c_sum = c_diff[sl].sum()/1e6
    t_rmse = np.sqrt(np.mean(t_diff[sl]**2))
    print(f"\n{label}:")
    print(f"  Heating diff (MWh): {h_sum:+.3f}")
    print(f"  Cooling diff (MWh): {c_sum:+.3f}")
    print(f"  Temp RMSE (°C):    {t_rmse:.4f}")



MONTHLY HEAT/COOL (MWh)
            py_heat_MWh  ml_heat_MWh  heat_diff  py_cool_MWh  ml_cool_MWh  \
2020-01-31        12.27        11.83       0.45         0.00         0.00   
2020-02-29        10.02         9.67       0.35         0.00         0.00   
2020-03-31         6.62         6.34       0.28         0.00         0.00   
2020-04-30         3.31         3.09       0.22         0.00         0.00   
2020-05-31         1.29         1.19       0.09         0.00         0.00   
2020-06-30         0.00         0.00       0.00         0.00         0.00   
2020-07-31         0.00         0.00       0.00        -0.23        -0.07   
2020-08-31         0.00         0.00       0.00        -0.31        -0.20   
2020-09-30         0.14         0.03       0.11         0.00         0.00   
2020-10-31         3.48         3.13       0.35         0.00         0.00   
2020-11-30         7.95         7.56       0.39         0.00         0.00   
2020-12-31        11.53        11.09       0.44    

/tmp/ipykernel_110020/1869460458.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  idx = pd.date_range("2020-01-01", periods=len(df_res_py), freq="H")
/tmp/ipykernel_110020/1869460458.py:8: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "py_heat_MWh": py['output_heating_power'].resample('M').sum()/1e6,
/tmp/ipykernel_110020/1869460458.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "ml_heat_MWh": mat['output_heating_power'].resample('M').sum()/1e6,
/tmp/ipykernel_110020/1869460458.py:10: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "py_cool_MWh": py['output_cooling_power'].resample('M').sum()/1e6,
/tmp/ipykernel_110020/1869460458.py:11: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "ml_cool_MWh": mat['output_co

In [6]:
# Diagnostics: net load deltas (heating deficit + extra cooling)
net_gain_diff_MWh = -(df_diff['output_heating_power'].sum()/1e6 + df_diff['output_cooling_power'].sum()/1e6)
net_gain_diff_Wavg = net_gain_diff_MWh * 1e6 / (len(df_res_py) * 3600)
print(f"\nNET GAIN DIFFERENCE (Python vs MATLAB)")
print(f"  Total surplus gains: {net_gain_diff_MWh:+.3f} MWh")
print(f"  Average surplus power: {net_gain_diff_Wavg:+.1f} W")

# Monthly net
df_month = df_diff[['output_heating_power','output_cooling_power']].copy()
df_month.index = pd.date_range("2020-01-01", periods=len(df_month), freq="H")
month_net = -(df_month.resample('M').sum().sum(axis=1))/1e6
print("\nMONTHLY NET GAIN DIFFERENCE (MWh):")
print(month_net.round(3))



NET GAIN DIFFERENCE (Python vs MATLAB)
  Total surplus gains: -2.412 MWh
  Average surplus power: -0.1 W

MONTHLY NET GAIN DIFFERENCE (MWh):
2020-01-31   -0.448
2020-02-29   -0.351
2020-03-31   -0.276
2020-04-30   -0.218
2020-05-31   -0.094
2020-06-30   -0.000
2020-07-31    0.160
2020-08-31    0.109
2020-09-30   -0.112
2020-10-31   -0.348
2020-11-30   -0.391
2020-12-31   -0.441
Freq: ME, dtype: float64


/tmp/ipykernel_110020/140772442.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_month.index = pd.date_range("2020-01-01", periods=len(df_month), freq="H")
/tmp/ipykernel_110020/140772442.py:11: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  month_net = -(df_month.resample('M').sum().sum(axis=1))/1e6
